# Limpieza de datos

La mayoría de la gente piensa que ser científico de datos significa estar corriendo modelos avanzados de Machine Learning todo el tiempo. La realidad es muy distinta:

![Tiempo de un científico de datos](./img/datascientist_time.jpeg)

La gran mayoría del tiempo se va **limpiando y organizando** los datos con los que queremos trabajar.

Los científicos de datos utilizamos herramientas como Pandas, Numpy, Matplotlib y Seaborn para limpiar los datos con los que queremos hacer algo.

## ETL

Un tipo de tarea que realizamos con gran frecuencia los científicos de datos son los **ETL**.

![Proceso de ETL](./img/etl.png)

ETL es un acrónimo que significa Extract, Transform, Load. Es un proceso que se utiliza para extraer datos de una fuente, transformarlos en un formato que sea adecuado para el análisis y cargarlos en una base de datos o algún otro sistema de almacenamiento.

## Ejemplo práctico

Los datos que usaremos para esta limpieza y nuestro siguiente análisis son datos de incidencia delictiva en nuestro país.

La iniciativa de datos abiertos del gobierno de México nos proporciona datos de incidencia delictiva desde 2015 hasta la fecha. Los datos se actualizan todos los meses y se pueden descargar desde el siguiente enlace: https://www.gob.mx/sesnsp/acciones-y-programas/datos-abiertos-de-incidencia-delictiva

---

Como podemos ver en el portal, se proporcionan los datos tanto a nivel estatal como a nivel municipal. En este caso, utilizaremos los datos a nivel estatal.

Descarguemos los datos y guardemos el archivo CSV en la carpeta data con el nombre `datos_delitos.csv`.

---

Ahora leamos el archivo CSV y veamos cómo se ven los datos.

Primero que nada, importemos pandas

In [39]:
import pandas as pd

In [40]:
#df = pd.read_csv('datos_delitos.csv')

Aquí tenemos un error muy común que suele ocurrir cuando un archivo se guarda en una computadora con cierto "encoding". 

Un encoding es una tabla que relaciona un número con un carácter. Por ejemplo, en la tabla ASCII, el número 65 corresponde a la letra "A".

Si el archivo que estamos leyendo fue guardado con un encoding distinto al que pandas espera, nos arrojará un error. 

Para solucionar esto, podemos utilizar el parámetro `encoding` de la función `pd.read_csv()` y especificar el encoding correcto.

¿Pero cómo sabemos con qué encoding cuenta el archivo?

La realidad es que la gran mayoría de los archivos los encontrarán en encoding utf-8 y Pandas no va a dar ningún error. Aquí estamos teniendo este problema porque la computadora que utilizan para generar este archivo uso un encoding diferente a utf-8.

En México, por lo general, si un archivo no está en utf-8, lo más seguro es que esté en `ISO-8859-1` o `latin1`.

Intentemos cargar el archivo especificando el encoding `ISO-8859-1`.

In [41]:
df = pd.read_csv('data/datos_delitos.csv', encoding='ISO-8859-1')
df.head()

,Año,Clave_Ent,Entidad,Bien jurídico afectado,Tipo de delito,Subtipo de delito,Modalidad,Enero,Febrero,Marzo,Abril,Mayo,Junio,Julio,Agosto,Septiembre,Octubre,Noviembre,Diciembre
0,2015,1,Aguascalientes,La vida y la Integridad corporal,Homicidio,Homicidio doloso,Con arma de fuego,3,0,2,1,1,1,2.0,1.0,2.0,2.0,2.0,1.0
1,2015,1,Aguascalientes,La vida y la Integridad corporal,Homicidio,Homicidio doloso,Con arma blanca,1,1,0,0,0,1,0.0,1.0,0.0,0.0,0.0,1.0
2,2015,1,Aguascalientes,La vida y la Integridad corporal,Homicidio,Homicidio doloso,Con otro elemento,0,0,2,2,3,2,0.0,1.0,2.0,0.0,0.0,0.0
3,2015,1,Aguascalientes,La vida y la Integridad corporal,Homicidio,Homicidio doloso,No especificado,2,0,0,1,0,0,0.0,0.0,0.0,0.0,0.0,0.0
4,2015,1,Aguascalientes,La vida y la Integridad corporal,Homicidio,Homicidio culposo,Con arma de fuego,0,0,0,0,1,0,0.0,0.0,0.0,0.0,0.0,0.0


Y vemos que ya podemos leer correctamente el archivo.


¿Existe alguna forma de verificar el encoding de un archivo sin tener que estar adivinando?

ChatGTP generó el siguiente código:

In [42]:
import chardet

def detect_encoding(file_path):
    with open(file_path, 'rb') as f:
        rawdata = f.read()
    result = chardet.detect(rawdata)
    return result

file_path = './data/datos_delitos.csv'
encoding_info = detect_encoding(file_path)
print(f"Detected encoding: {encoding_info['encoding']}")

Detected encoding: Windows-1252


Sigamos...

In [43]:
df.tail(3)

,Año,Clave_Ent,Entidad,Bien jurídico afectado,Tipo de delito,Subtipo de delito,Modalidad,Enero,Febrero,Marzo,Abril,Mayo,Junio,Julio,Agosto,Septiembre,Octubre,Noviembre,Diciembre
31357,2024,32,Zacatecas,Otros bienes jurídicos afectados (del fuero co...,Delitos cometidos por servidores públicos,Delitos cometidos por servidores públicos,Delitos cometidos por servidores públicos,20,22,34,35,34,37,NaN,NaN,NaN,NaN,NaN,NaN
31358,2024,32,Zacatecas,Otros bienes jurídicos afectados (del fuero co...,Electorales,Electorales,Electorales,0,1,0,11,31,23,NaN,NaN,NaN,NaN,NaN,NaN
31359,2024,32,Zacatecas,Otros bienes jurídicos afectados (del fuero co...,Otros delitos del Fuero Común,Otros delitos del Fuero Común,Otros delitos del Fuero Común,151,183,174,199,208,177,NaN,NaN,NaN,NaN,NaN,NaN


In [44]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 31360 entries, 0 to 31359
Data columns (total 19 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   Año                     31360 non-null  int64  
 1   Clave_Ent               31360 non-null  int64  
 2   Entidad                 31360 non-null  str    
 3   Bien jurídico afectado  31360 non-null  str    
 4   Tipo de delito          31360 non-null  str    
 5   Subtipo de delito       31360 non-null  str    
 6   Modalidad               31360 non-null  str    
 7   Enero                   31360 non-null  int64  
 8   Febrero                 31360 non-null  int64  
 9   Marzo                   31360 non-null  int64  
 10  Abril                   31360 non-null  int64  
 11  Mayo                    31360 non-null  int64  
 12  Junio                   31360 non-null  int64  
 13  Julio                   28224 non-null  float64
 14  Agosto                  28224 non-null  float64
 

Bien. Están muy bien los datos. Sin embargo, tenemos un problema con el que es muy común encontrarnos. 

Resulta que el formato en el que está el archivo es fácil entender por seres humanos:


```markdown
| Año      | Entidad  | Enero    | Febrero  | Mes X    |
|----------|----------|----------|----------|----------|
|   ...    |   ...    |   ...    |   ...    |   ...    |
|   ...    |   ...    |   ...    |   ...    |   ...    |
|   ...    |   ...    |   ...    |   ...    |   ...    |
|   ...    |   ...    |   ...    |   ...    |   ...    |
|   ...    |   ...    |   ...    |   ...    |   ...    |
```

Vemos que tenemos los meses como encabezados. Es decir, el archivo está tratando cada mes como si fuera una variable. 

Nosotros como científicos de datos estamos más interesados en conjuntos de datos que no estén en este formato de "resumen" o "tabla dinámica". Para nosotros, lo ideal sería que cada mes fuera simplemente una observación más en nuestro conjunto de datos. Es decir, queremos transformar la tabla de arriba en:

```markdown
| Año      | Entidad  | Mes        |
|----------|----------|------------|
|   ...    |   ...    |   Enero    |
|   ...    |   ...    |   Febrero  |
|   ...    |   ...    |   Marzo    |
|   ...    |   ...    |   Abril    |
|   ...    |   ...    |   Mes X    |
```

A este tipo de formato le llamos "formato largo de datos". 

Haremos las siguientes limpiezas:
* Transformar los nombres de las columnas para que no tengan caracteres especiales y estén siempre en minúsculas
* Convertir el dataset a un formato de datos "largo"

In [45]:
for col in df.columns:
    print(col)

Año
Clave_Ent
Entidad
Bien jurídico afectado
Tipo de delito
Subtipo de delito
Modalidad
Enero
Febrero
Marzo
Abril
Mayo
Junio
Julio
Agosto
Septiembre
Octubre
Noviembre
Diciembre


In [46]:
len(df.columns)

19

### Limpiar nombres de columnas

In [47]:
def limpiar_columnas(df):
    columnas_limpias = []
    for col in df.columns:
        # convertir a minusculas, reemplazar espacios por guiones bajos y eliminar caracteres especiales
        col = col.lower().replace(" ", "_").replace("ñ", "ni").replace(".", "").replace("á", "a").replace("é", "e").replace("í","i").replace("ó", "o").replace("ú", "u")
        columnas_limpias.append(col)
    
    df.columns = columnas_limpias
    
    return df


In [48]:
df.head(2)

,Año,Clave_Ent,Entidad,Bien jurídico afectado,Tipo de delito,Subtipo de delito,Modalidad,Enero,Febrero,Marzo,Abril,Mayo,Junio,Julio,Agosto,Septiembre,Octubre,Noviembre,Diciembre
0,2015,1,Aguascalientes,La vida y la Integridad corporal,Homicidio,Homicidio doloso,Con arma de fuego,3,0,2,1,1,1,2.0,1.0,2.0,2.0,2.0,1.0
1,2015,1,Aguascalientes,La vida y la Integridad corporal,Homicidio,Homicidio doloso,Con arma blanca,1,1,0,0,0,1,0.0,1.0,0.0,0.0,0.0,1.0


In [49]:
df = limpiar_columnas(df)

In [50]:
df.head(3)

,anio,clave_ent,entidad,bien_juridico_afectado,tipo_de_delito,subtipo_de_delito,modalidad,enero,febrero,marzo,abril,mayo,junio,julio,agosto,septiembre,octubre,noviembre,diciembre
0,2015,1,Aguascalientes,La vida y la Integridad corporal,Homicidio,Homicidio doloso,Con arma de fuego,3,0,2,1,1,1,2.0,1.0,2.0,2.0,2.0,1.0
1,2015,1,Aguascalientes,La vida y la Integridad corporal,Homicidio,Homicidio doloso,Con arma blanca,1,1,0,0,0,1,0.0,1.0,0.0,0.0,0.0,1.0
2,2015,1,Aguascalientes,La vida y la Integridad corporal,Homicidio,Homicidio doloso,Con otro elemento,0,0,2,2,3,2,0.0,1.0,2.0,0.0,0.0,0.0


Muy bien. Ahora lo que queremos hacer es quitar algunas columnas. Nos interesan nada más las siguientes:

In [51]:
df[['anio', 'entidad']]

,anio,entidad
0,2015,Aguascalientes
1,2015,Aguascalientes
2,2015,Aguascalientes
3,2015,Aguascalientes
4,2015,Aguascalientes
...,...,...
31355,2024,Zacatecas
31356,2024,Zacatecas
31357,2024,Zacatecas
31358,2024,Zacatecas


In [52]:
df = df[['anio', 'clave_ent', 'entidad', 'tipo_de_delito', 'subtipo_de_delito', 'modalidad','enero', 'febrero', 'marzo', 'abril', 'mayo', 'junio', 'julio', 'agosto', 'septiembre', 'octubre', 'noviembre', 'diciembre']]
df.head(2)

,anio,clave_ent,entidad,tipo_de_delito,subtipo_de_delito,modalidad,enero,febrero,marzo,abril,mayo,junio,julio,agosto,septiembre,octubre,noviembre,diciembre
0,2015,1,Aguascalientes,Homicidio,Homicidio doloso,Con arma de fuego,3,0,2,1,1,1,2.0,1.0,2.0,2.0,2.0,1.0
1,2015,1,Aguascalientes,Homicidio,Homicidio doloso,Con arma blanca,1,1,0,0,0,1,0.0,1.0,0.0,0.0,0.0,1.0


### Formato largo de datos

Ahora, usaremos el método `melt` para convertir las columnas a observaciones.

Queremos convervar las coumnas:
* anio
* clave_ent
* entidad
* tipo_de_delito
* subtipo_de_delito
* modalidad

El resto de las columnas las vamos a juntar en una nueva columna llamada "nombre_mes" y sus valores los vamos a sumar en otra llamada "frecuencia"

In [53]:
print("Shape ", df.shape)

Shape  (31360, 18)


In [54]:
datos_long = df.melt(id_vars=['anio', 'clave_ent', 'entidad','tipo_de_delito', 'subtipo_de_delito', 'modalidad'], var_name='nombre_mes', value_name='frecuencia')

In [55]:
print("Shape: ", datos_long.shape)

Shape:  (376320, 8)


In [56]:
datos_long.head(5)

,anio,clave_ent,entidad,tipo_de_delito,subtipo_de_delito,modalidad,nombre_mes,frecuencia
0,2015,1,Aguascalientes,Homicidio,Homicidio doloso,Con arma de fuego,enero,3.0
1,2015,1,Aguascalientes,Homicidio,Homicidio doloso,Con arma blanca,enero,1.0
2,2015,1,Aguascalientes,Homicidio,Homicidio doloso,Con otro elemento,enero,0.0
3,2015,1,Aguascalientes,Homicidio,Homicidio doloso,No especificado,enero,2.0
4,2015,1,Aguascalientes,Homicidio,Homicidio culposo,Con arma de fuego,enero,0.0


In [57]:
datos_long.info()

<class 'pandas.DataFrame'>
RangeIndex: 376320 entries, 0 to 376319
Data columns (total 8 columns):
 #   Column             Non-Null Count   Dtype  
---  ------             --------------   -----  
 0   anio               376320 non-null  int64  
 1   clave_ent          376320 non-null  int64  
 2   entidad            376320 non-null  str    
 3   tipo_de_delito     376320 non-null  str    
 4   subtipo_de_delito  376320 non-null  str    
 5   modalidad          376320 non-null  str    
 6   nombre_mes         376320 non-null  str    
 7   frecuencia         357504 non-null  float64
dtypes: float64(1), int64(2), str(5)
memory usage: 23.0 MB


Supongamos que para este análisis, no nos importan los niveles subtipo de delito y modalidad. O sea, no queremos tener la distinción entre homicidios dolosos y culposos (sé que son bastante diferentes, pero simplifiquemos nuestro ejemplo).

Vamos a agrupar nuestro dataframe por anio, clave_ent, entidad, tipo_de_delito y nombre_mes. Esto hará que todos los tipos de homicidios se sumen al tipo "homicidio" o todos los tipos de robo de vehículo (con o sin violencia) se sumen a "robo de vehículo".

In [58]:
datos_long = datos_long.groupby(['anio', 'clave_ent', 'entidad', 'tipo_de_delito', 'nombre_mes'])['frecuencia'].sum().reset_index()

In [59]:
datos_long[datos_long.tipo_de_delito == 'Robo'].sample(15)

,anio,clave_ent,entidad,tipo_de_delito,nombre_mes,frecuencia
91114,2020,30,Veracruz de Ignacio de la Llave,Robo,octubre,2069.0
33509,2017,6,Colima,Robo,julio,713.0
87755,2020,23,Quintana Roo,Robo,septiembre,1250.0
58466,2018,26,Sonora,Robo,diciembre,600.0
37347,2017,14,Jalisco,Robo,enero,7780.0
16707,2016,3,Baja California Sur,Robo,enero,909.0
97832,2021,12,Guerrero,Robo,mayo,557.0
136235,2023,28,Tamaulipas,Robo,septiembre,822.0
34468,2017,8,Chihuahua,Robo,febrero,1336.0
85824,2020,19,Nuevo León,Robo,abril,1202.0


Mostremos todos los estados y su respectiva clave

In [60]:
datos_long[['clave_ent', 'entidad']].drop_duplicates().sort_values('entidad')

,clave_ent,entidad
0,1,Aguascalientes
480,2,Baja California
960,3,Baja California Sur
1440,4,Campeche
2880,7,Chiapas
3360,8,Chihuahua
3840,9,Ciudad de México
1920,5,Coahuila de Zaragoza
2400,6,Colima
4320,10,Durango


Ahora Veamos todos los datos de delitos de una entidad en específico. Por ejemplo, Nuevo león

In [61]:
datos_long[datos_long['clave_ent'] == 19].sample(15)

,anio,clave_ent,entidad,tipo_de_delito,nombre_mes,frecuencia
70131,2019,19,Nuevo León,Allanamiento de morada,enero,23.0
131720,2023,19,Nuevo León,Feminicidio,mayo,7.0
116397,2022,19,Nuevo León,Hostigamiento sexual,noviembre,12.0
131578,2023,19,Nuevo León,Allanamiento de morada,octubre,50.0
85627,2020,19,Nuevo León,Falsificación,marzo,74.0
24367,2016,19,Nuevo León,Otros delitos que atentan contra la vida y la ...,marzo,2.0
101010,2021,19,Nuevo León,Fraude,junio,807.0
24476,2016,19,Nuevo León,Violencia familiar,mayo,1618.0
116560,2022,19,Nuevo León,Secuestro,febrero,2.0
55046,2018,19,Nuevo León,Otros delitos del Fuero Común,diciembre,235.0


In [62]:
datos_long[(datos_long['clave_ent'] > 19) &  (datos_long['clave_ent'] < 24) & (datos_long['tipo_de_delito'] == 'Homicidio')].sample(15)

,anio,clave_ent,entidad,tipo_de_delito,nombre_mes,frecuencia
87101,2020,22,Querétaro,Homicidio,julio,40.0
9823,2015,21,Puebla,Homicidio,marzo,103.0
101498,2021,20,Oaxaca,Homicidio,diciembre,157.0
26147,2016,23,Quintana Roo,Homicidio,septiembre,26.0
118302,2022,23,Quintana Roo,Homicidio,junio,125.0
116861,2022,20,Oaxaca,Homicidio,julio,146.0
101981,2021,21,Puebla,Homicidio,julio,108.0
55426,2018,20,Oaxaca,Homicidio,octubre,132.0
148540,2024,22,Querétaro,Homicidio,febrero,42.0
72221,2019,23,Quintana Roo,Homicidio,julio,133.0


### Valores de fechas

Finalmente, queremos tener una columna "fecha". Actualmente tenemos el año y el nombre del mes, pero no tenemos como tal una columna que tenga un tipo de dato fecha. Eso hace que filtrar por fecha sea complicado.

Por ejemplo, si queremos conocer todos los homicidios de Oaxaca en enero 2024, haríamos lo siguiente:

In [63]:
datos_long[
    (datos_long['clave_ent'] == 20) &
    (datos_long['tipo_de_delito'] == 'Homicidio') &
    (datos_long['anio'] == 2024) &
    (datos_long['nombre_mes'] == 'enero') 
]

,anio,clave_ent,entidad,tipo_de_delito,nombre_mes,frecuencia
147579,2024,20,Oaxaca,Homicidio,enero,157.0


Creemos una columna de fecha. 

Primero tenemos que convertir el nombre de mes a un número, en donde 1 es enero, 2 febrero, etc.

In [64]:
datos_long.sample(2)

,anio,clave_ent,entidad,tipo_de_delito,nombre_mes,frecuencia
73834,2019,26,Sonora,Robo,octubre,763.0
13896,2015,29,Tlaxcala,Violencia de género en todas sus modalidades d...,abril,0.0


In [65]:
datos_long["nueva_columna"] = "dato vacío"
datos_long.sample(6)

,anio,clave_ent,entidad,tipo_de_delito,nombre_mes,frecuencia,nueva_columna
14996,2015,32,Zacatecas,Delitos cometidos por servidores públicos,mayo,17.0,dato vacío
46482,2018,1,Aguascalientes,Secuestro,junio,1.0,dato vacío
38978,2017,18,Nayarit,Daño a la propiedad,diciembre,3.0,dato vacío
29711,2016,30,Veracruz de Ignacio de la Llave,Tráfico de menores,septiembre,0.0,dato vacío
35399,2017,10,Durango,Otros delitos que atentan contra la libertad y...,septiembre,5.0,dato vacío
124476,2023,4,Campeche,Extorsión,abril,11.0,dato vacío


In [66]:
# Diccionario de ayuda para convertir
meses = {
    "enero": 1,
    "febrero": 2,
    "marzo": 3,
    "abril": 4,
    "mayo": 5,
    "junio": 6,
    "julio": 7,
    "agosto": 8,
    "septiembre": 9,
    "octubre": 10,
    "noviembre": 11,
    "diciembre": 12
}

datos_long['mes'] = datos_long['nombre_mes'].map(meses)
datos_long.sample(5)

,anio,clave_ent,entidad,tipo_de_delito,nombre_mes,frecuencia,nueva_columna,mes
98290,2021,13,Hidalgo,Otros delitos que atentan contra la vida y la ...,octubre,8.0,dato vacío,10
109562,2022,5,Coahuila de Zaragoza,Despojo,diciembre,26.0,dato vacío,12
114233,2022,14,Jalisco,Violencia familiar,julio,1042.0,dato vacío,7
15602,2016,1,Aguascalientes,Incesto,diciembre,0.0,dato vacío,12
32449,2017,4,Campeche,Otros delitos contra el patrimonio,agosto,1.0,dato vacío,8


In [67]:
datos_long["frecuencia_mas_10"] = datos_long["frecuencia"] + 10
datos_long.sample(5)

,anio,clave_ent,entidad,tipo_de_delito,nombre_mes,frecuencia,nueva_columna,mes,frecuencia_mas_10
21637,2016,14,Jalisco,Acoso sexual,agosto,6.0,dato vacío,8,16.0
71868,2019,22,Querétaro,Otros delitos que atentan contra la libertad y...,abril,4.0,dato vacío,4,14.0
114396,2022,15,México,Extorsión,abril,460.0,dato vacío,4,470.0
19851,2016,10,Durango,Falsedad,enero,0.0,dato vacío,1,10.0
126074,2023,7,Chiapas,Otros delitos contra la sociedad,diciembre,7.0,dato vacío,12,17.0


In [68]:
datos_long["anio_mes"] = datos_long["anio"].astype(str) + datos_long["mes"].astype(str)
datos_long.sample(6)

,anio,clave_ent,entidad,tipo_de_delito,nombre_mes,frecuencia,nueva_columna,mes,frecuencia_mas_10,anio_mes
84456,2020,16,Michoacán de Ocampo,Violencia de género en todas sus modalidades d...,abril,0.0,dato vacío,4,10.0,20204
7985,2015,17,Morelos,Otros delitos contra la familia,julio,0.0,dato vacío,7,10.0,20157
42368,2017,25,Sinaloa,Despojo,mayo,25.0,dato vacío,5,35.0,20175
126994,2023,9,Ciudad de México,Lesiones,octubre,1024.0,dato vacío,10,1034.0,202310
40177,2017,20,Oaxaca,Otros delitos que atentan contra la libertad p...,agosto,21.0,dato vacío,8,31.0,20178
147045,2024,19,Nuevo León,Extorsión,noviembre,0.0,dato vacío,11,10.0,202411


In [69]:
# yyyy-mm-dd, yy-mm-dd, yymmdd, yyyy/dd/mm

In [70]:
# Agregamos la columna de fecha juntando el año y el mes
datos_long['fecha'] = pd.to_datetime(datos_long['anio'].astype(str) + datos_long['mes'].astype(str), format='%Y%m')
datos_long.sample(5)

,anio,clave_ent,entidad,tipo_de_delito,nombre_mes,frecuencia,nueva_columna,mes,frecuencia_mas_10,anio_mes,fecha
123350,2023,1,Aguascalientes,Violencia familiar,diciembre,376.0,dato vacío,12,386.0,202312,2023-12-01
40859,2017,22,Querétaro,Allanamiento de morada,septiembre,15.0,dato vacío,9,25.0,20179,2017-09-01
20694,2016,12,Guerrero,Allanamiento de morada,junio,18.0,dato vacío,6,28.0,20166,2016-06-01
145651,2024,16,Michoacán de Ocampo,Fraude,marzo,220.0,dato vacío,3,230.0,20243,2024-03-01
67160,2019,12,Guerrero,Violación equiparada,mayo,13.0,dato vacío,5,23.0,20195,2019-05-01


In [71]:
# Eliminamos las columnas que ya no necesitamos
datos_long = datos_long.drop(columns=['nueva_columna'])
datos_long.sample(5)

,anio,clave_ent,entidad,tipo_de_delito,nombre_mes,frecuencia,mes,frecuencia_mas_10,anio_mes,fecha
41846,2017,24,San Luis Potosí,Corrupción de menores,diciembre,2.0,12,12.0,201712,2017-12-01
117330,2022,21,Puebla,Fraude,junio,439.0,6,449.0,20226,2022-06-01
4280,2015,9,Ciudad de México,Violación equiparada,mayo,8.0,5,18.0,20155,2015-05-01
22509,2016,15,México,Tráfico de menores,noviembre,0.0,11,10.0,201611,2016-11-01
153418,2024,32,Zacatecas,Otros delitos contra el patrimonio,octubre,0.0,10,10.0,202410,2024-10-01


Veamos los homicidios en oaxaca de enero 2024 a la fecha

In [78]:
datos_long[
    (datos_long.tipo_de_delito == "Homicidio") &
    (datos_long.clave_ent == 20) &
    (datos_long.fecha >= '2024-01-01')
].sort_values('fecha')

,anio,clave_ent,entidad,tipo_de_delito,nombre_mes,frecuencia,mes,frecuencia_mas_10,anio_mes,fecha
147579,2024,20,Oaxaca,Homicidio,enero,157.0,1,167.0,20241,2024-01-01
147580,2024,20,Oaxaca,Homicidio,febrero,170.0,2,180.0,20242,2024-02-01
147583,2024,20,Oaxaca,Homicidio,marzo,190.0,3,200.0,20243,2024-03-01
147576,2024,20,Oaxaca,Homicidio,abril,208.0,4,218.0,20244,2024-04-01
147584,2024,20,Oaxaca,Homicidio,mayo,236.0,5,246.0,20245,2024-05-01
147582,2024,20,Oaxaca,Homicidio,junio,202.0,6,212.0,20246,2024-06-01
147581,2024,20,Oaxaca,Homicidio,julio,0.0,7,10.0,20247,2024-07-01
147577,2024,20,Oaxaca,Homicidio,agosto,0.0,8,10.0,20248,2024-08-01
147587,2024,20,Oaxaca,Homicidio,septiembre,0.0,9,10.0,20249,2024-09-01
147586,2024,20,Oaxaca,Homicidio,octubre,0.0,10,10.0,202410,2024-10-01


In [73]:
datos_finales = datos_long[['anio', 'clave_ent', 'entidad', 'tipo_de_delito', 'nombre_mes', 'fecha', 'frecuencia']]
datos_finales.head(2)

,anio,clave_ent,entidad,tipo_de_delito,nombre_mes,fecha,frecuencia
0,2015,1,Aguascalientes,Aborto,abril,2015-04-01,0.0
1,2015,1,Aguascalientes,Aborto,agosto,2015-08-01,0.0


Ya que tenemos muestros datos bien estructurados, los podemos guardar en nuestra computadora. Los guardaremos con el nombre "delitos.csv"

In [ ]:
datos_finales.to_csv('data/delitos.csv', index=False)